<a href="https://colab.research.google.com/github/krishnayaswanthp-netizen/Feb-2026-Internship-Innomatics-Research-Lab/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Load Model & Tokenizer
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Fix padding issue (important)
tokenizer.pad_token = tokenizer.eos_token

# 2. Initialize chat history
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

# 3. Chat Loop
while True:

    # Take user input
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    # 4. Tokenize input with attention mask
    inputs = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')

    new_input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    # 5. Maintain conversation history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # 6. Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # 7. Decode only new response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # 8. Print response
    print("Chatbot:", response)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: hello
Chatbot: I'm a girl
You: how are you


RuntimeError: The size of tensor a (11) must match the size of tensor b (4) at non-singleton dimension 1